In [6]:
import pandas as pd
from nltk.tokenize import word_tokenize
from collections import defaultdict

ot_words = []
nt_words = []
word_to_target = defaultdict(list)
# Dictionary to store word -> source text mapping
word_to_source = defaultdict(list)

ot = pd.read_csv("../ebible-corpus/dhao/aligned-ind-dhao-ot.csv")
nt = pd.read_csv("../ebible-corpus/dhao/aligned-ind-dhao-nt.csv")
# ot = ot[ot["verse"].str.contains("GEN")]

for idx, row in ot.iterrows():
    target_text = row["target_text"]
    normalized_tokens = word_tokenize(row["source_text"].replace('“', '"'))
    normalized_tokens = word_tokenize(row["source_text"].replace('’', "'"))
    clean_tokens = [word.lower() for word in normalized_tokens if word.isalpha()]
    for word in clean_tokens:
        word_to_target[word].append(target_text)
        word_to_source[word].append(row["source_text"])
    ot_words.extend(clean_tokens)
    
for idx, row in nt.iterrows():
    target_text = row["target_text"]
    normalized_tokens = word_tokenize(row["source_text"].replace('“', '"'))
    normalized_tokens = word_tokenize(row["source_text"].replace('’', "'"))
    clean_tokens = [word.lower() for word in normalized_tokens if word.isalpha()]
    nt_words.extend(clean_tokens)
    
# Only keep words in nt_words that are not in ot_words
nt_words_set = set(nt_words)
ot_words = [w for w in ot_words if w not in nt_words_set]

oov_word_to_target = {word: targets for word, targets in word_to_target.items() if word not in nt_words_set}
oov_word_to_source = {word: sources for word, sources in word_to_source.items() if word not in nt_words_set}

In [7]:
import matplotlib.pyplot as plt
from collections import Counter

word_counts = Counter(ot_words)

# Create DataFrame with word, freq_count, target_text, and source_text columns
oov_data = []
for word, count in word_counts.most_common():
    # Get the first occurrence of the word's target and source text
    target_text = oov_word_to_target[word][0] if word in oov_word_to_target else ""
    source_text = oov_word_to_source[word][0] if word in oov_word_to_source else ""
    
    oov_data.append({
        'word': word, 
        'freq_count': count,
        'target_text': target_text,
        'source_text': source_text
    })

oov_freq_df = pd.DataFrame(oov_data)

# Save to CSV
output_path = "../ebible-corpus/dhao/ot_oov_freq_count.csv"
oov_freq_df.to_csv(output_path, index=False)

print(f"Saved {len(oov_freq_df)} OOV words to {output_path}")
print(f"Top 10 most frequent OOV words:")
print(oov_freq_df.head(10))

Saved 1008 OOV words to ../ebible-corpus/dhao/ot_oov_freq_count.csv
Top 10 most frequent OOV words:
        word  freq_count                                        target_text  \
0      abram          71  Lodꞌo umur Tera dꞌai pidhu nguru tèu risi, na ...   
1      laban          62  Ana mone Ribka, ngara na Laban. Ropa Laban lad...   
2        lea          36  Laban ne, dhu dènge ana bhèni dhèu dua. Dhu ur...   
3  abimelekh          27  Hèia na peka dènge dhèu ètu sèra, aku nèngu na...   
4        het          22  Ana uuru Kanaꞌan, nuka, Sidon. Kanaꞌan jꞌajꞌi ...   
5     ismael          22  Èu dhu dènge babia. Nèbhu heka, èu mora iisi a...   
6       adik          22  Hèia ra hia Ribka dènge bhèni dhu leru ne karè...   
7      sarai          18  Abram madhèdi dènge Sarai. Nahor leo dènge ana...   
8   jenisnya          17  Ka Ama Lamatua lii hari, peka na, “Rai hudꞌi p...   
9   tendanya          13  Ca tèka, na ninu èi anggor, ka na mahu titu kè...   

                              